# ShuffleRec: enriched features and chronological evaluation

This notebook reads the master table and normalized listening history. Model code
lives in `shufflerec/` so preprocessing, training, evaluation and inference share
one implementation. Install `requirements-model.txt` in this notebook kernel.

The expected schema includes the metadata and acoustic columns from
`extracting_features`. Missing optional columns are supported and reported.
Explicit status is excluded. Source tables and acquisition backlogs are never
written. Training artifacts stay under ignored `sensitive_info/model_runs/`.

Training, multi-model comparisons and final testing require their separate flags
below. Running the notebook with the defaults prepares data and reports parameter
counts, without starting training or inspecting test performance.


In [7]:
import sys
from pathlib import Path
import json
import torch

current = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (current, current.parent) if (p / "shufflerec").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the project root or notebooks/")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shufflerec.audit import audit_paths, read_catalog, read_history, signature
from shufflerec.data import prepare_data
from shufflerec.features import FeatureConfig
from shufflerec.model import ModelConfig, SpotifySASRec
from shufflerec.training import build_evaluation_plan, evaluate, fit, seed_everything, save_checkpoint, load_checkpoint
from shufflerec.experiments import run_comparison

MASTER_DB = PROJECT_ROOT / "data/master_track_table.csv"
HISTORY_DB = PROJECT_ROOT / "sensitive_info/spotify_listening_history.csv"


## Settings

The default candidate universe is tracks in the current catalog that also occur
in the supplied history. This is a snapshot of your personal listening catalog;
it includes held-out item identities, but their future listening outcomes are
not used to fit preprocessing or train. Set `CANDIDATE_SCOPE = "catalog"` to
use the full supplied catalog (substantially more memory and preparation time).
Sampled Hit@10/NDCG@10 describe this candidate protocol, not full-catalog ranking.

Artist embeddings have 8 dimensions; genres have 4; time signatures have 2.
Release year describes the represented release/reissue, not necessarily the
original recording. Local acoustic inputs keep their own names, local-source
flags and measured analysis durations; unknown analysis scope stays unknown.


In [ ]:
RUN_AUDIT = False
RUN_TRAINING = True
RUN_COMPARISON = False
RUN_FINAL_TEST = False

CANDIDATE_SCOPE = "history"
SEED = 42
MAX_LENGTH = 10
NUM_NEGATIVES = 99
MAX_EVAL_EXAMPLES = 500

FEATURE_CONFIG = FeatureConfig()
MODEL_CONFIG = ModelConfig()

TRAINING_OPTIONS = dict(
    epochs=5,
    preference_epochs=3,
    batch_size=32,
    learning_rate=0.001,
    preference_learning_rate=0.0001,
    patience=3,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", DEVICE)


Training device: cuda


## Read-only audit and preparation

Complete sessions are assigned chronologically, approximately 80/10/10 by session
count. Contexts never cross partition boundaries. Vocabulary, aliases and numerical
statistics fit training-session tracks only. The test summary below reports counts,
not prediction metrics. Metadata-only tracks can participate; unknown labels and
missing labels remain distinct from sequence padding.


In [9]:
if CANDIDATE_SCOPE not in ("history", "catalog"):
    raise ValueError("CANDIDATE_SCOPE must be history or catalog")
if RUN_AUDIT:
    print(json.dumps(audit_paths(MASTER_DB, HISTORY_DB), indent=2))
history = read_history(HISTORY_DB)
track_ids = set(history.track_id.dropna().str.strip()) if CANDIDATE_SCOPE == "history" else None
catalog = read_catalog(MASTER_DB, track_ids=track_ids)
data = prepare_data(catalog, history, FEATURE_CONFIG, max_length=MAX_LENGTH)
print(json.dumps(data.summary(), indent=2))
seed_everything(SEED)
model = SpotifySASRec(data.preprocessor, MAX_LENGTH, MODEL_CONFIG).to(DEVICE)
print(json.dumps(model.parameter_report(), indent=2))
print("Candidate scope:", CANDIDATE_SCOPE, "| usable tracks:", len(data.store.usable_ids))


{
  "catalog_rows": 1374357,
  "duplicate_ids": 0,
  "history_events": 310705,
  "unique_history_tracks": 8170,
  "matched_history_tracks": 2107,
  "missing_history_tracks": 6063,
  "matched_history_events": 107674,
  "valid_numeric_counts": {
    "danceability": 1371423,
    "energy": 1374059,
    "loudness": 1374357,
    "speechiness": 1289037,
    "acousticness": 1289037,
    "instrumentalness": 1374357,
    "liveness": 1289037,
    "valence": 1289037,
    "tempo": 1371424,
    "mode": 1374357,
    "key_sin": 1374357,
    "key_cos": 1374357,
    "year": 1289015,
    "duration_ms": 1374036,
    "local_rms_db": 0,
    "local_onset_strength": 0,
    "local_spectral_centroid": 0,
    "local_spectral_contrast": 0,
    "local_beat_regularity": 0,
    "tempo_is_local": 0,
    "tempo_analysis_seconds": 0,
    "key_is_local": 0,
    "key_analysis_seconds": 0,
    "mode_is_local": 0,
    "mode_analysis_seconds": 0,
    "local_rms_db_is_local": 0,
    "local_rms_db_analysis_seconds": 0,
    "l

## Train and select using validation only

Both next-item and skip-pair optimization use training sessions. Validation NDCG
selects weights and supports early stopping. Preference tuning is retained only
when it improves that same validation criterion. The fixed candidate plan uses
seeded examples throughout the validation period, rather than just its first rows.
Model/preprocessing artifacts are new files; old experiments are never replaced.


In [10]:
training_result = None
validation_plan = None
run_dir = None
if RUN_TRAINING:
    validation_plan = build_evaluation_plan(data.examples["validation"]["nip"], data.store.usable_ids,
        num_negatives=NUM_NEGATIVES, max_examples=MAX_EVAL_EXAMPLES, seed=SEED,
        scope=CANDIDATE_SCOPE + " catalog snapshot")
    training_result = fit(model, data, validation_plan, seed=SEED,
                          logger=lambda row: print(json.dumps(row)), **TRAINING_OPTIONS)
    from datetime import datetime, timezone
    from uuid import uuid4
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
    run_dir = PROJECT_ROOT / "sensitive_info/model_runs" / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    metadata = dict(training=training_result, partitions=data.summary(), candidate_scope=CANDIDATE_SCOPE,
                    master_signature=signature(MASTER_DB), history_signature=signature(HISTORY_DB),
                    model_parameters=model.parameter_report(), training_options=TRAINING_OPTIONS)
    save_checkpoint(run_dir / "model.pt", model, data.preprocessor, metadata=metadata)
    with (run_dir / "training_summary.json").open("x", encoding="utf-8") as stream:
        json.dump(metadata, stream, indent=2)
    print("Selected validation metrics:", training_result["best_validation"])
    print("Saved private model artifacts:", run_dir)
else:
    print("Preparation complete. Review counts and parameters before enabling RUN_TRAINING.")


{"phase": "next_item", "epoch": 1, "loss": 0.5784646316726121, "validation": {"hit_at_k": 0.2, "ndcg_at_k": 0.09502929610801993, "k": 10, "examples": 100, "mean_candidates": 100.0, "min_candidates": 100, "max_candidates": 100, "candidate_scope": "history catalog snapshot", "seed": 42}}
{"phase": "preference", "epoch": 1, "loss": 0.7756146102919614, "validation": {"hit_at_k": 0.19, "ndcg_at_k": 0.0853194148473265, "k": 10, "examples": 100, "mean_candidates": 100.0, "min_candidates": 100, "max_candidates": 100, "candidate_scope": "history catalog snapshot", "seed": 42}}
Selected validation metrics: {'hit_at_k': 0.2, 'ndcg_at_k': 0.09502929610801993, 'k': 10, 'examples': 100, 'mean_candidates': 100.0, 'min_candidates': 100, 'max_candidates': 100, 'candidate_scope': 'history catalog snapshot', 'seed': 42}
Saved private model artifacts: D:\BetterSpotifyShuffle\sensitive_info\model_runs\20261019T212754Z-315e6e51


## Optional feature comparisons

Enable separately: four independent fits compare audio-only, added metadata,
compact key/time-signature encoding, and added local acoustics/provenance. Every
variant uses the same common validation targets and candidate lists; the report
also shows differing training-example coverage. The test partition is not scored.
Album and popularity can be added explicitly through custom `FeatureConfig`s.


In [11]:
comparison_results = None
if RUN_COMPARISON:
    comparison_results = run_comparison(catalog, history, model_config=MODEL_CONFIG,
        max_length=MAX_LENGTH, epochs=TRAINING_OPTIONS["epochs"],
        preference_epochs=TRAINING_OPTIONS["preference_epochs"], batch_size=TRAINING_OPTIONS["batch_size"],
        num_negatives=NUM_NEGATIVES, max_examples=MAX_EVAL_EXAMPLES, seed=SEED,
        device=DEVICE, logger=lambda row: print(json.dumps(row)))
    print(json.dumps(comparison_results, indent=2))


## Final test: enable only after choosing the configuration

Leave this disabled while comparing features/settings. It evaluates the selected
model on later sessions with a separate fixed candidate plan. Do not choose
another model based on this result and continue calling the same partition an
untouched test set. Current metadata snapshots cannot reconstruct all historical
availability or historical popularity.


In [12]:
test_metrics = None
if RUN_FINAL_TEST:
    if training_result is None:
        raise ValueError("Train/select a model first, or load a chosen checkpoint explicitly")
    test_plan = build_evaluation_plan(data.examples["test"]["nip"], data.store.usable_ids,
        num_negatives=NUM_NEGATIVES, max_examples=MAX_EVAL_EXAMPLES, seed=SEED + 1,
        scope=CANDIDATE_SCOPE + " catalog snapshot")
    test_metrics = evaluate(model, data.store, test_plan)
    print(json.dumps(test_metrics, indent=2))
    with (run_dir / "final_test.json").open("x", encoding="utf-8") as stream:
        json.dump(test_metrics, stream, indent=2)


## Reusing a checkpoint

`loaded_model, fitted_preprocessor, metadata = load_checkpoint(path, device=DEVICE)`
loads the feature configuration, frozen vocabulary/scaling, architecture and
weights together. Build a `TrackStore` from the inference catalog using that
preprocessor; do not refit it. Encode past context through `store.context`, then
use `loaded_model.score_candidates` for candidate batches. Legacy 24-input
checkpoints require the legacy implementation and are not compatible here.

The notebook intentionally contains no backlog-writing utility or Spotify calls.
